# 05b — Train CutScorer on Colab (T4 GPU)

**Feature leakage fix:** `cut_delay`, `area_flow`, `required_time`, `slack` are
excluded from training features. They are used only for label construction
(`mffc_combined`). The model now learns from 9 structural/topological features only.

**Before running:**
1. Set runtime to GPU: `Runtime → Change runtime type → T4 GPU`
2. Upload `ml/train_data.npz` (from `05a_preprocess_mac.py`) to Google Drive at:
   `My Drive/ml_cut_project/train_data.npz`
3. **Run Cell 6b first** to clear old checkpoints (N_FEATURES changed from 14 → 9)


## Cell 1 — Imports & GPU check

In [ ]:
import os, time, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_PROJECT_DIR = '/content/drive/MyDrive/ml_cut_project'
NPZ_PATH          = os.path.join(DRIVE_PROJECT_DIR, 'train_data.npz')
CKPT_DIR          = os.path.join(DRIVE_PROJECT_DIR, 'checkpoints')
FINAL_MODEL_PATH  = os.path.join(DRIVE_PROJECT_DIR, 'cut_model_mlp.pt')
SCALER_PKL_PATH   = os.path.join(DRIVE_PROJECT_DIR, 'scaler.pkl')

os.makedirs(CKPT_DIR, exist_ok=True)

if not os.path.exists(NPZ_PATH):
    raise FileNotFoundError(
        f'ERROR: {NPZ_PATH} not found.\n'
        'Upload ml/train_data.npz from your Mac to '
        'My Drive/ml_cut_project/train_data.npz first.'
    )

print(f'Drive mounted  ✓')
print(f'NPZ found      : {NPZ_PATH}  ({os.path.getsize(NPZ_PATH)/1024:.0f} KB)')
print(f'Checkpoint dir : {CKPT_DIR}')

## Cell 3 — Config  *(must stay in sync with `05a_preprocess_mac.py`)*

In [ ]:
# 9 structural features — label-only cols (cut_delay, area_flow, required_time, slack) excluded
FEATURE_NAMES = [
    'n_leaves',
    'node_level',
    'node_fanout',
    'is_critical',
    'slack_ratio',
    'fanout_adj_area',
    'area_per_leaf',
    'mffc_size',
    'mffc_per_leaf',
]
N_FEATURES = len(FEATURE_NAMES)   # 9
HIDDEN     = [64, 32]

EPOCHS       = 60
BATCH_SIZE   = 8192
LR           = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT      = 0.1
LR_PATIENCE  = 8
SEED         = 42
CKPT_EVERY   = 2

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'Config : {N_FEATURES} features (structural only)  |  hidden={HIDDEN}  |  dropout={DROPOUT}  |  wd={WEIGHT_DECAY}  |  patience={LR_PATIENCE}  |  {EPOCHS} epochs')
print(f'NOTE: cut_delay, area_flow, required_time, slack excluded — label leakage fix')


## Cell 4 — Load preprocessed data

In [ ]:
print(f'Loading {NPZ_PATH} ...')
data = np.load(NPZ_PATH, allow_pickle=True)

X_scaled        = data['X_scaled']         # (N, 9) already scaled
pair_better     = data['pair_better']       # (P,)
pair_worse      = data['pair_worse']        # (P,)
circuits        = data['circuits']          # (N,)
unique_circuits = data['unique_circuits']
quality         = data['quality']           # (N,)
is_mffc_best    = data['adj_score_best']    # (N,) bool
is_best         = data['is_best']           # (N,) int8 — ABC's choice
scaler_mean     = data['scaler_mean']       # (9,)
scaler_scale    = data['scaler_scale']      # (9,)

print(f'  Rows            : {len(X_scaled):,}')
print(f'  Pairs           : {len(pair_better):,}')
print(f'  Circuits        : {unique_circuits.tolist()}')
print(f'  Feature shape   : {X_scaled.shape}')

assert X_scaled.shape[1] == N_FEATURES, (
    f'ERROR: NPZ has {X_scaled.shape[1]} features but N_FEATURES={N_FEATURES}. '
    f'Re-run 05a_preprocess_mac.py and re-upload train_data.npz.'
)
print('Data loaded ✓')


## Cell 5 — Train / val split  *(same SEED as Mac)*

In [ ]:
n_circuits = len(unique_circuits)

if n_circuits == 1:
    val_circuits   = set(unique_circuits.tolist())
    train_circuits = set(unique_circuits.tolist())
else:
    rng      = np.random.default_rng(SEED)
    shuffled = rng.permutation(unique_circuits)
    n_val          = max(1, int(n_circuits * 0.2))
    val_circuits   = set(shuffled[:n_val].tolist())
    train_circuits = set(shuffled[n_val:].tolist())

print(f'Train circuits ({len(train_circuits)}): {sorted(train_circuits)}')
print(f'Val   circuits ({len(val_circuits)}):   {sorted(val_circuits)}')

pair_circs = circuits[pair_better]
train_mask = np.array([c in train_circuits for c in pair_circs])
val_mask   = ~train_mask

pb_train = pair_better[train_mask];  pw_train = pair_worse[train_mask]
pb_val   = pair_better[val_mask];    pw_val   = pair_worse[val_mask]

print(f'Train pairs : {len(pb_train):,}')
print(f'Val   pairs : {len(pb_val):,}')

if len(pb_train) == 0:
    raise RuntimeError('ERROR: No training pairs — check train_data.npz.')

## Cell 6 — Model definition

In [ ]:
class CutScorer(nn.Module):
    """
    N_FEATURES → ReLU → Dropout → 64 → ReLU → Dropout → 32 → 1
    FIX: in_dim now taken from N_FEATURES (14), not hardcoded 12.
    Dropout is only active during model.train(); no-op at eval/export.
    """
    def __init__(self, in_dim=N_FEATURES, hidden=HIDDEN, dropout=DROPOUT):
        super().__init__()
        self.fc0   = nn.Linear(in_dim,    hidden[0])
        self.drop0 = nn.Dropout(dropout)
        self.fc1   = nn.Linear(hidden[0], hidden[1])
        self.drop1 = nn.Dropout(dropout)
        self.fc2   = nn.Linear(hidden[1], 1)

    def forward(self, x):
        h = self.drop0(F.relu(self.fc0(x)))
        h = self.drop1(F.relu(self.fc1(h)))
        return self.fc2(h).squeeze(-1)   # raw logit

    def score_sigmoid(self, x):
        return torch.sigmoid(self.forward(x))


def ranknet_loss(model, x_better, x_worse):
    """RankNet: push score(better) > score(worse) via BCE on score difference."""
    diff = model(x_better) - model(x_worse)
    return F.binary_cross_entropy_with_logits(diff, torch.ones_like(diff))


print('CutScorer defined  ✓')
print(f'Architecture: {N_FEATURES} → ReLU+Drop({DROPOUT}) → {HIDDEN[0]} → ReLU+Drop({DROPOUT}) → {HIDDEN[1]} → 1')

## Cell 6b — Clear old checkpoints for fresh retrain
*(Skip this cell if you want to RESUME a previous run)*

In [ ]:
# SKIP THIS CELL if you want to resume rather than restart.
import shutil
old_ckpts = [
    os.path.join(CKPT_DIR, f)
    for f in os.listdir(CKPT_DIR)
    if f.startswith('ckpt_epoch') and f.endswith('.pt')
]
for c in old_ckpts:
    os.remove(c)
    print(f'Removed: {os.path.basename(c)}')
if not old_ckpts:
    print('No old checkpoints found — clean start.')
else:
    print(f'Cleared {len(old_ckpts)} old checkpoint(s).  Starting fresh.')

## Cell 7 — Resume from checkpoint or start fresh

In [ ]:
def latest_checkpoint(ckpt_dir):
    ckpts = sorted([
        f for f in os.listdir(ckpt_dir)
        if f.startswith('ckpt_epoch') and f.endswith('.pt')
    ])
    return os.path.join(ckpt_dir, ckpts[-1]) if ckpts else None


model     = CutScorer().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, patience=LR_PATIENCE, factor=0.5, verbose=False)

start_epoch   = 1
best_val_loss = float('inf')
best_state    = None

ckpt_path = latest_checkpoint(CKPT_DIR)
if ckpt_path:
    print(f'Resuming from checkpoint: {ckpt_path}')
    ckpt = torch.load(ckpt_path, map_location=device)
    # Verify checkpoint feature count matches current config
    ckpt_n_feat = ckpt.get('n_features', None)
    if ckpt_n_feat is not None and ckpt_n_feat != N_FEATURES:
        raise ValueError(
            f'Checkpoint has n_features={ckpt_n_feat} but current N_FEATURES={N_FEATURES}. '
            'Run Cell 6b to clear old checkpoints first.'
        )
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    start_epoch   = ckpt['epoch'] + 1
    best_val_loss = ckpt['best_val_loss']
    best_state    = ckpt['best_model_state']
    print(f'  Resumed at epoch {start_epoch}  (best val loss so far: {best_val_loss:.4f})')
else:
    print('No checkpoint found — starting from scratch.')

if start_epoch > EPOCHS:
    print(f'Training already complete ({EPOCHS} epochs).')

## Cell 8 — Training loop

In [ ]:
X_t = torch.from_numpy(X_scaled)   # CPU tensor; batches moved to GPU per step

print(f'Training epochs {start_epoch}–{EPOCHS}  |  batch={BATCH_SIZE}  |  ckpt every {CKPT_EVERY} epochs')
print('-' * 72)

t_start = time.time()

for epoch in range(start_epoch, EPOCHS + 1):

    # Train
    model.train()
    perm      = np.random.permutation(len(pb_train))
    pb_shuf   = pb_train[perm]
    pw_shuf   = pw_train[perm]
    ep_loss   = 0.0
    n_batches = 0

    for start in range(0, len(pb_shuf), BATCH_SIZE):
        end = start + BATCH_SIZE
        xb  = X_t[pb_shuf[start:end]].to(device)
        xw  = X_t[pw_shuf[start:end]].to(device)
        optimizer.zero_grad()
        loss = ranknet_loss(model, xb, xw)
        loss.backward()
        optimizer.step()
        ep_loss   += loss.item()
        n_batches += 1

    avg_train = ep_loss / max(n_batches, 1)

    # Validate
    avg_val = 0.0
    if len(pb_val) > 0:
        model.eval()
        with torch.no_grad():
            vl = []
            for start in range(0, len(pb_val), BATCH_SIZE * 4):
                end = start + BATCH_SIZE * 4
                xb  = X_t[pb_val[start:end]].to(device)
                xw  = X_t[pw_val[start:end]].to(device)
                vl.append(ranknet_loss(model, xb, xw).item())
            avg_val = float(np.mean(vl))

    scheduler.step(avg_val if avg_val > 0 else avg_train)
    lr      = optimizer.param_groups[0]['lr']
    elapsed = time.time() - t_start

    flag = ''
    if avg_val > 0 and avg_val < best_val_loss:
        best_val_loss = avg_val
        best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        flag = '  ← best'

    print(f'  Epoch {epoch:2d}/{EPOCHS}  train={avg_train:.4f}  '
          f'val={avg_val:.4f}  lr={lr:.1e}  {elapsed:.0f}s{flag}')

    if epoch % CKPT_EVERY == 0 or epoch == EPOCHS:
        ckpt_file = os.path.join(CKPT_DIR, f'ckpt_epoch{epoch:03d}.pt')
        torch.save({
            'epoch'            : epoch,
            'model_state'      : {k: v.cpu() for k, v in model.state_dict().items()},
            'optimizer_state'  : optimizer.state_dict(),
            'scheduler_state'  : scheduler.state_dict(),
            'best_val_loss'    : best_val_loss,
            'best_model_state' : best_state,
            'train_loss'       : avg_train,
            'val_loss'         : avg_val,
            'hidden_sizes'     : HIDDEN,
            'n_features'       : N_FEATURES,
            'feature_names'    : FEATURE_NAMES,
        }, ckpt_file)
        print(f'             💾 Saved checkpoint → {os.path.basename(ckpt_file)}')

        all_ckpts = sorted([
            os.path.join(CKPT_DIR, f)
            for f in os.listdir(CKPT_DIR)
            if f.startswith('ckpt_epoch') and f.endswith('.pt')
        ])
        for old in all_ckpts[:-3]:
            os.remove(old)

print(f'\nBest val loss : {best_val_loss:.4f}')

## Cell 9 — Evaluation on validation pairs

In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)
    print('Loaded best model weights for evaluation.')

model.eval()
with torch.no_grad():
    xb_val = X_t[pb_val].to(device)
    xw_val = X_t[pw_val].to(device)
    s_b    = model(xb_val).cpu()
    s_w    = model(xw_val).cpu()
    pair_acc = (s_b > s_w).float().mean().item()

# How often does ABC agree with our MFFC-best label?
# FIX: is_mffc_best replaces adj_score_best here
abc_mffc_agree = float(is_mffc_best[is_best == 1].mean())

print(f'Pair ranking accuracy (val)       : {pair_acc:.1%}')
print(f'  (fraction of val pairs correctly ordered by model)')
print(f'ABC picks MFFC-optimal cut        : {abc_mffc_agree:.1%}')
print(f'  (lower = more room for ML to beat ABC)')
print()
if pair_acc < 0.55:
    print('WARNING: pair accuracy near 50% — model has learned little.')
    print('  Check that scaler looks healthy and mffc_size is in the data.')
elif pair_acc > 0.70:
    print('✓ Model is ranking cuts significantly better than random.')

## Cell 10 — Save final model + scaler to Drive

In [ ]:
state_to_save = best_state or {k: v.cpu() for k, v in model.state_dict().items()}

torch.save({
    'model_state'  : state_to_save,
    'hidden_sizes' : HIDDEN,
    'n_features'   : N_FEATURES,
    'feature_names': FEATURE_NAMES,
}, FINAL_MODEL_PATH)
print(f'✓ cut_model_mlp.pt saved → {FINAL_MODEL_PATH}')
print(f'  ({os.path.getsize(FINAL_MODEL_PATH)/1024:.1f} KB)')

# Reconstruct scaler so 06_export_weights_to_c.py works on Mac
sc = StandardScaler()
sc.mean_          = scaler_mean.astype(np.float64)
sc.scale_         = scaler_scale.astype(np.float64)
sc.var_           = sc.scale_ ** 2
sc.n_features_in_ = N_FEATURES
sc.n_samples_seen_= 1

with open(SCALER_PKL_PATH, 'wb') as f:
    pickle.dump({'scaler': sc, 'feature_names': FEATURE_NAMES}, f)
print(f'✓ scaler.pkl saved → {SCALER_PKL_PATH}')

## Cell 11 — Next steps on your Mac

1. **Download** `cut_model_mlp.pt` from `My Drive/ml_cut_project/` → `ml/cut_model_mlp.pt`

2. ```bash
   python3 06_export_weights_to_c.py
   ```
3. ```bash
   ./07_install_ml_into_abc.sh
   ```
4. ```bash
   python3 08_compare_qor.py
   ```
